# Apache Airflow: Orchestrating ML Pipelines

## What Is Airflow?

Imagine you need to cook a multi-course dinner: first make soup, then roast chicken, then bake dessert.  
Each dish depends on the previous one finishing. You can't serve dessert before dinner.  
**Apache Airflow** is like a smart kitchen manager that schedules all these tasks in the right order,  
retries if something goes wrong, and alerts you if a dish burns.

In ML terms: Airflow schedules and monitors your data pipelines — extract data → clean it → train model → evaluate → deploy.

## Why Airflow?

Without a workflow tool, ML teams run scripts manually or with cron jobs.  
Problems: no retry on failure, no monitoring, no dependency management, no history of past runs.

Airflow solves all of these.

## Resources

- **Docs**: [https://airflow.apache.org/docs/](https://airflow.apache.org/docs/)
- **GitHub**: [https://github.com/apache/airflow](https://github.com/apache/airflow)
- **YouTube — Airflow Tutorial**: [https://www.youtube.com/watch?v=K9AnJ9_ZAXE](https://www.youtube.com/watch?v=K9AnJ9_ZAXE)
- **YouTube — Airflow for ML**: [https://www.youtube.com/watch?v=s-r2gEr7YW4](https://www.youtube.com/watch?v=s-r2gEr7YW4)

## Installation

```bash
# Install with a constraint file to avoid dependency conflicts
AIRFLOW_VERSION=2.8.1
PYTHON_VERSION=3.11
pip install apache-airflow==${AIRFLOW_VERSION} \
    --constraint https://raw.githubusercontent.com/apache/airflow/constraints-${AIRFLOW_VERSION}/constraints-${PYTHON_VERSION}.txt

# Initialize the database and start everything
airflow standalone
# → Web UI at http://localhost:8080 (admin/admin)
```

For this notebook, all Airflow code runs **without a running Airflow instance**.  
We import and define DAGs as Python objects — the same code you'd deploy to a real Airflow server.

In [ ]:
from datetime import datetime, timedelta
import json

try:
    from airflow import DAG
    from airflow.operators.python import PythonOperator
    from airflow.operators.bash import BashOperator
    from airflow.operators.empty import EmptyOperator
    from airflow.utils.dates import days_ago
    AIRFLOW_AVAILABLE = True
    print("Airflow available — full functionality enabled")
except ImportError:
    AIRFLOW_AVAILABLE = False
    print("Airflow not installed — showing code structure with simulated output")
    print("Install: pip install apache-airflow")

print("\nAll DAG definitions below are valid Airflow code.")
print("Copy any DAG to your Airflow DAGs folder (~airflow/dags/) to deploy it.")

## Core Concept 1: DAGs — Directed Acyclic Graphs

A **DAG** (Directed Acyclic Graph) is a workflow definition.  
- **Directed**: tasks flow in one direction (A → B → C)
- **Acyclic**: no loops (you can't have A → B → A)
- **Graph**: tasks are nodes, dependencies are edges

Each DAG has:
- A unique `dag_id`
- A `schedule_interval` (cron expression or preset like `@daily`)
- A `start_date` (when to start scheduling)
- Default arguments applied to all tasks

### Cron Expressions
```
┌───── minute (0-59)
│ ┌───── hour (0-23)
│ │ ┌───── day of month (1-31)
│ │ │ ┌───── month (1-12)
│ │ │ │ ┌───── day of week (0-6, Sun=0)
│ │ │ │ │
0 2 * * *    → Every day at 2:00 AM
0 */6 * * *  → Every 6 hours
@daily       → Once per day at midnight
@weekly      → Once per week
```

In [ ]:
# ── Define default arguments shared by all tasks in the DAG ──────────────────
default_args = {
    'owner': 'ml_team',
    'depends_on_past': False,      # don't wait for previous day's run to succeed
    'email': ['mlops@company.com'],
    'email_on_failure': True,      # alert on task failure
    'email_on_retry': False,
    'retries': 2,                  # retry failed tasks up to 2 times
    'retry_delay': timedelta(minutes=5),  # wait 5 min between retries
}

print("Default args (applied to all tasks):")
print(json.dumps({k: str(v) for k, v in default_args.items()}, indent=2))
print()
print("Common schedule_interval values:")
schedules = [
    ("@once",    "Run once only"),
    ("@hourly",  "Every hour (0 * * * *)"),
    ("@daily",   "Every day at midnight (0 0 * * *)"),
    ("@weekly",  "Every Sunday at midnight"),
    ("@monthly", "First day of every month"),
    ("None",     "No automatic scheduling (trigger manually)"),
    ("0 2 * * 1-5", "2:00 AM Monday-Friday (typical ETL)"),
]
for s, desc in schedules:
    print(f"  {s:20s} → {desc}")

## Core Concept 2: Operators — Task Types

Each task in a DAG is an **Operator**. Airflow has many built-in operators:

| Operator | Use Case |
|----------|----------|
| `PythonOperator` | Run any Python function |
| `BashOperator` | Run a shell command |
| `EmptyOperator` | Placeholder / grouping task |
| `EmailOperator` | Send an email |
| `HttpOperator` | Make an HTTP request |
| `S3ToRedshiftOperator` | AWS data transfer |
| `BigQueryOperator` | Run SQL on BigQuery |
| `SparkSubmitOperator` | Submit a Spark job |

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import pickle, os, tempfile

# ── Functions that will become Airflow tasks ──────────────────────────────────
DATA_PATH  = os.path.join(tempfile.gettempdir(), 'airflow_demo_data.pkl')
MODEL_PATH = os.path.join(tempfile.gettempdir(), 'airflow_demo_model.pkl')

def ingest_data(**context):
    """Task 1: Simulate pulling data from a database or API."""
    print("[ingest_data] Fetching data from source...")
    X, y = make_classification(n_samples=2000, n_features=20,
                                n_informative=10, random_state=42)
    df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(20)])
    df['label'] = y
    df.to_pickle(DATA_PATH)
    print(f"[ingest_data] Saved {len(df)} rows to {DATA_PATH}")
    # Push metadata to XCom so downstream tasks can use it
    if AIRFLOW_AVAILABLE:
        context['ti'].xcom_push(key='n_rows', value=len(df))
    return len(df)


def preprocess_data(**context):
    """Task 2: Clean and preprocess the data."""
    print("[preprocess_data] Loading and cleaning data...")
    df = pd.read_pickle(DATA_PATH)

    # Simulate preprocessing
    df = df.dropna()
    # Clip outliers to 3 standard deviations
    feature_cols = [c for c in df.columns if c != 'label']
    for col in feature_cols:
        mean, std = df[col].mean(), df[col].std()
        df[col] = df[col].clip(mean - 3*std, mean + 3*std)

    df.to_pickle(DATA_PATH)
    print(f"[preprocess_data] Preprocessed {len(df)} rows")
    return {"n_rows": len(df), "n_features": len(feature_cols)}


def train_model(**context):
    """Task 3: Train the ML model."""
    print("[train_model] Training model...")
    df = pd.read_pickle(DATA_PATH)
    X = df.drop('label', axis=1).values
    y = df['label'].values
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    val_acc = accuracy_score(y_val, model.predict(X_val))
    val_auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])

    with open(MODEL_PATH, 'wb') as f:
        pickle.dump(model, f)

    print(f"[train_model] val_accuracy={val_acc:.4f}  val_auc={val_auc:.4f}")
    if AIRFLOW_AVAILABLE:
        context['ti'].xcom_push(key='val_auc', value=val_auc)
    return {"val_accuracy": val_acc, "val_auc": val_auc}


def evaluate_model(**context):
    """Task 4: Evaluate the model against the previous production model."""
    print("[evaluate_model] Evaluating model...")
    if AIRFLOW_AVAILABLE:
        val_auc = context['ti'].xcom_pull(key='val_auc', task_ids='train_model')
    else:
        with open(MODEL_PATH, 'rb') as f:
            model = pickle.load(f)
        df = pd.read_pickle(DATA_PATH)
        X = df.drop('label', axis=1).values
        y = df['label'].values
        val_auc = roc_auc_score(y, model.predict_proba(X)[:, 1])

    PRODUCTION_AUC_THRESHOLD = 0.85  # minimum AUC to deploy
    if val_auc >= PRODUCTION_AUC_THRESHOLD:
        print(f"[evaluate_model] PASS: val_auc={val_auc:.4f} >= {PRODUCTION_AUC_THRESHOLD}")
        return "deploy"
    else:
        print(f"[evaluate_model] FAIL: val_auc={val_auc:.4f} < {PRODUCTION_AUC_THRESHOLD}")
        raise ValueError(f"Model quality too low: AUC={val_auc:.4f}")


def deploy_model(**context):
    """Task 5: Deploy the model to production."""
    print("[deploy_model] Deploying model to production API...")
    # In real life: copy to S3, update model registry, restart serving pod, etc.
    import shutil
    deploy_path = os.path.join(tempfile.gettempdir(), 'production_model.pkl')
    shutil.copy(MODEL_PATH, deploy_path)
    print(f"[deploy_model] Model deployed to {deploy_path}")
    print("[deploy_model] Sending deployment notification to Slack...")


# Run the pipeline manually to verify all functions work
print("Running ML pipeline manually (simulating Airflow execution):")
print("-" * 60)
context = {}  # empty context (no XCom in manual run)
ingest_data(**context)
preprocess_data(**context)
train_model(**context)
evaluate_model(**context)
deploy_model(**context)
print("-" * 60)
print("All pipeline tasks completed successfully!")

## Core Concept 3: Defining the DAG

Now we assemble the Python functions into an Airflow DAG with task dependencies.

In [ ]:
if AIRFLOW_AVAILABLE:
    with DAG(
        dag_id='ml_retraining_pipeline',
        default_args=default_args,
        description='Daily ML model retraining pipeline',
        schedule_interval='0 2 * * *',  # every day at 2 AM
        start_date=datetime(2024, 1, 1),
        catchup=False,          # don't run for all past dates
        tags=['ml', 'retraining', 'production'],
    ) as dag:

        start = EmptyOperator(task_id='start')

        ingest = PythonOperator(
            task_id='ingest_data',
            python_callable=ingest_data,
        )

        preprocess = PythonOperator(
            task_id='preprocess_data',
            python_callable=preprocess_data,
        )

        train = PythonOperator(
            task_id='train_model',
            python_callable=train_model,
        )

        evaluate = PythonOperator(
            task_id='evaluate_model',
            python_callable=evaluate_model,
        )

        deploy = PythonOperator(
            task_id='deploy_model',
            python_callable=deploy_model,
        )

        end = EmptyOperator(task_id='end')

        # ── Define execution order with >> operator ──────────────────────────
        start >> ingest >> preprocess >> train >> evaluate >> deploy >> end

    print(f"DAG '{dag.dag_id}' defined with {len(dag.tasks)} tasks")
    print("Tasks:", [t.task_id for t in dag.topological_sort()])
else:
    print("DAG definition (valid Airflow code — install Airflow to run):")
    print()
    print("  with DAG('ml_retraining_pipeline',")
    print("           schedule_interval='0 2 * * *',")
    print("           start_date=datetime(2024, 1, 1),")
    print("           catchup=False) as dag:")
    print()
    print("      start      = EmptyOperator(task_id='start')")
    print("      ingest     = PythonOperator(task_id='ingest_data',    python_callable=ingest_data)")
    print("      preprocess = PythonOperator(task_id='preprocess_data', python_callable=preprocess_data)")
    print("      train      = PythonOperator(task_id='train_model',    python_callable=train_model)")
    print("      evaluate   = PythonOperator(task_id='evaluate_model', python_callable=evaluate_model)")
    print("      deploy     = PythonOperator(task_id='deploy_model',   python_callable=deploy_model)")
    print("      end        = EmptyOperator(task_id='end')")
    print()
    print("      start >> ingest >> preprocess >> train >> evaluate >> deploy >> end")
    print()
    print("  Task execution order:")
    print("  start → ingest_data → preprocess_data → train_model → evaluate_model → deploy_model → end")

## Core Concept 4: XComs — Passing Data Between Tasks

**XComs** (Cross-Communications) let tasks share small pieces of data.

**Important**: XComs are stored in Airflow's database — keep them small (IDs, metrics, file paths).  
For large data (DataFrames, models), write to S3/disk and pass the path via XCom.

```python
# Push value
context['ti'].xcom_push(key='model_accuracy', value=0.93)

# Pull value (from another task)
acc = context['ti'].xcom_pull(key='model_accuracy', task_ids='train_model')
```

**Anti-pattern**: Passing a 1GB DataFrame via XCom will crash your database!

In [ ]:
# Demonstrate XCom pattern correctly
print("XCom patterns: RIGHT vs WRONG")
print()

print("WRONG — passing large data via XCom:")
print("""
  def task_a(**context):
      df = pd.read_csv('huge_file.csv')  # 1 GB DataFrame
      context['ti'].xcom_push(key='data', value=df)  # CRASH: too large for DB

  def task_b(**context):
      df = context['ti'].xcom_pull(key='data', task_ids='task_a')  # Error!
""")

print("RIGHT — pass file path via XCom, read data from filesystem/S3:")
print("""
  def task_a(**context):
      df = pd.read_csv('huge_file.csv')
      path = '/tmp/processed_data.parquet'
      df.to_parquet(path)  # save to filesystem
      context['ti'].xcom_push(key='data_path', value=path)  # just a string!

  def task_b(**context):
      path = context['ti'].xcom_pull(key='data_path', task_ids='task_a')
      df = pd.read_parquet(path)  # load from filesystem
      # ... process df ...
""")

print("XCom best practices:")
for tip in [
    "Push only small values: strings, numbers, short dicts/lists",
    "For files: push the path, read the file in the downstream task",
    "Use S3/GCS for large artifacts (models, DataFrames)",
    "XCom values are visible in the Airflow UI under task instance details",
    "Default XCom: return value of python_callable is auto-pushed as 'return_value'",
]:
    print(f"  • {tip}")

## Core Concept 5: Task Dependencies — Branching and Fan-out

Tasks don't have to be linear. You can fan-out (parallel tasks) and fan-in (wait for multiple):

In [ ]:
# Demonstrate complex dependency patterns
print("Airflow dependency patterns:")
print()

patterns = {
    "Linear": (
        "t1 >> t2 >> t3",
        "t1 → t2 → t3"
    ),
    "Fan-out (parallel)": (
        "t1 >> [t2, t3, t4]",
        "t1 → t2, t1 → t3, t1 → t4  (t2/t3/t4 run in parallel)"
    ),
    "Fan-in (join)": (
        "[t1, t2, t3] >> t4",
        "t1, t2, t3 must all finish before t4 starts"
    ),
    "Diamond": (
        "t1 >> [t2, t3] >> t4",
        "t1 → t2 and t3 in parallel → t4 waits for both"
    ),
    "Branching": (
        "BranchPythonOperator: return task_id to run",
        "Dynamically choose which path to take based on runtime data"
    ),
}

for name, (code, description) in patterns.items():
    print(f"  {name}:")
    print(f"    Code:    {code}")
    print(f"    Effect:  {description}")
    print()

# Branching example
print("Branching example — deploy to different environments:")
print("""
  from airflow.operators.python import BranchPythonOperator

  def choose_environment(**context):
      day = context['execution_date'].weekday()  # 0=Mon, 6=Sun
      if day < 5:  # weekday
          return 'deploy_to_staging'
      else:        # weekend
          return 'skip_deployment'

  branch = BranchPythonOperator(
      task_id='choose_env',
      python_callable=choose_environment,
  )

  branch >> [deploy_staging, skip]
""")

## Common Pitfalls

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Importing heavy modules at DAG level | Scheduler slow/crashes | Import inside task functions, not at top of DAG file |
| XCom with large data | Database out of space | Store data externally, pass only paths |
| `catchup=True` with old start_date | Thousands of runs triggered | Set `catchup=False` or use a recent start_date |
| Mutable default args | Same dict shared across DAGs | Use `default_args` factory function |
| Not setting `depends_on_past=False` | Pipeline stuck waiting forever | Set False unless you explicitly need sequential backfills |
| Complex logic in DAG file | Hard to test | Keep DAG file thin, import functions from separate modules |
| No retries configured | Transient failures stop pipeline | Set `retries=2` and `retry_delay` in default_args |

## Mini Project: Complete ML Retraining DAG

A production-ready DAG that:
1. Checks if new data is available
2. Ingests, cleans, and validates it
3. Trains and evaluates a model
4. Only deploys if model beats the champion
5. Sends Slack notification

In [ ]:
# Complete production-ready DAG
# This code is deployment-ready — save as a .py file in airflow/dags/

PRODUCTION_DAG_CODE = '''
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator, BranchPythonOperator
from airflow.operators.empty import EmptyOperator
import pandas as pd
import numpy as np
import pickle, os

# ── Configuration ─────────────────────────────────────────────────────────────
DATA_PATH        = '/data/ml/daily_data.parquet'
CHAMPION_PATH    = '/models/champion.pkl'
CHALLENGER_PATH  = '/models/challenger.pkl'
MIN_AUC          = 0.85

default_args = {
    'owner': 'ml_team',
    'retries': 3,
    'retry_delay': timedelta(minutes=10),
    'email_on_failure': True,
    'email': ['mlops@company.com'],
}

# ── Task functions ────────────────────────────────────────────────────────────
def check_data_availability(**context):
    """Check if enough new data exists. Branch accordingly."""
    run_date = context["ds"]  # execution date as string e.g. "2024-01-15"
    # In real code: query database, check S3 file exists, etc.
    if os.path.exists(DATA_PATH):
        return "ingest_data"     # proceed with training
    else:
        return "skip_no_data"    # not enough data today, skip

def ingest_data(**context):
    # Load from database/S3
    df = pd.read_parquet(DATA_PATH)
    context["ti"].xcom_push(key="n_rows", value=len(df))

def validate_data(**context):
    n = context["ti"].xcom_pull(key="n_rows", task_ids="ingest_data")
    assert n >= 1000, f"Too few rows: {n} < 1000"
    assert n < 10_000_000, f"Suspiciously many rows: {n}"  # data pipeline bug?

def train_model(**context):
    from sklearn.ensemble import GradientBoostingClassifier
    from sklearn.model_selection import cross_val_score
    df = pd.read_parquet(DATA_PATH)
    X, y = df.drop("label", axis=1).values, df["label"].values
    model = GradientBoostingClassifier(n_estimators=100)
    cv_auc = cross_val_score(model, X, y, cv=5, scoring="roc_auc").mean()
    model.fit(X, y)
    with open(CHALLENGER_PATH, "wb") as f: pickle.dump(model, f)
    context["ti"].xcom_push(key="challenger_auc", value=cv_auc)

def promote_if_better(**context):
    """Promote challenger to champion only if it beats the threshold."""
    challenger_auc = context["ti"].xcom_pull(key="challenger_auc", task_ids="train_model")
    if challenger_auc >= MIN_AUC:
        import shutil
        shutil.copy(CHALLENGER_PATH, CHAMPION_PATH)
        return "notify_success"
    return "notify_model_degraded"

def notify_success(**context): print("Model promoted! Slack: ✅ New champion deployed")
def notify_degraded(**context): print("Model quality low. Slack: ⚠️ Keeping old champion")
def skip_handler(**context): print("Not enough data today, skipping.")

# ── DAG Definition ────────────────────────────────────────────────────────────
with DAG(
    dag_id="ml_daily_retraining",
    default_args=default_args,
    schedule_interval="0 3 * * *",  # 3 AM daily
    start_date=datetime(2024, 1, 1),
    catchup=False,
    tags=["ml", "production", "daily"],
    doc_md="""## ML Daily Retraining Pipeline
    Checks for new data, trains a challenger model, and promotes it
    if it meets quality threshold."""
) as dag:

    check_data = BranchPythonOperator(
        task_id="check_data_availability",
        python_callable=check_data_availability)

    skip_no_data = PythonOperator(
        task_id="skip_no_data",
        python_callable=skip_handler)

    ingest  = PythonOperator(task_id="ingest_data",   python_callable=ingest_data)
    validate= PythonOperator(task_id="validate_data", python_callable=validate_data)
    train   = PythonOperator(task_id="train_model",   python_callable=train_model)

    promote = BranchPythonOperator(
        task_id="promote_if_better",
        python_callable=promote_if_better)

    notify_ok  = PythonOperator(task_id="notify_success",       python_callable=notify_success)
    notify_bad = PythonOperator(task_id="notify_model_degraded",python_callable=notify_degraded)

    done = EmptyOperator(task_id="done", trigger_rule="none_failed_min_one_success")

    # Dependencies
    check_data >> [skip_no_data, ingest]
    ingest >> validate >> train >> promote
    promote >> [notify_ok, notify_bad]
    [skip_no_data, notify_ok, notify_bad] >> done
'''

print("Production ML retraining DAG (save as airflow/dags/ml_retraining.py):")
print(PRODUCTION_DAG_CODE)

## Interview Questions and Answers

In [ ]:
qa = [
    {"q": "What is a DAG in Airflow? Why must it be acyclic?",
     "a": """A DAG (Directed Acyclic Graph) is a workflow where tasks are nodes and dependencies are directed edges.
It must be acyclic (no loops) because:
1. A cycle (A→B→A) would mean the pipeline never finishes
2. Airflow's scheduler topologically sorts tasks to find the execution order — cycles make this impossible
3. Real workflows are always finite: you extract data, then process, then store — never in a loop

If you need to run a pipeline repeatedly, that's handled by the schedule_interval (run the whole DAG daily)
not by looping within the DAG."""},

    {"q": "Airflow vs Prefect vs Luigi — when would you choose each?",
     "a": """Apache Airflow:
- Most mature and widely adopted (de facto standard)
- Excellent UI for monitoring and debugging
- DAGs defined as static Python files (evaluated at schedule time)
- Complex to set up (requires database + scheduler + workers)
- Best for: large teams, complex pipelines, heavy monitoring needs

Prefect:
- Modern, Python-first API (no DAG class, just decorators)
- Dynamic workflows possible (DAG structure determined at runtime)
- Easier to test locally, better developer experience
- Cloud version (Prefect Cloud) very polished
- Best for: teams wanting simpler code, dynamic pipelines

Luigi:
- Older, from Spotify, simple and lightweight
- Target-based: tasks define what they output, Luigi figures out what to run
- No UI comparable to Airflow
- Best for: simple pipelines, teams that prefer target-based logic

Choose Airflow when the company already uses it or you need enterprise monitoring.
Choose Prefect for new projects wanting developer-friendliness."""},

    {"q": "What is idempotency and why does it matter in Airflow?",
     "a": """Idempotency: running the same task multiple times produces the same result.

Why it matters: Airflow retries failed tasks. If your task is not idempotent,
retries can corrupt data or double-insert rows.

Bad (not idempotent):
  INSERT INTO predictions SELECT ... FROM raw_data
  # If retried: rows are INSERTED TWICE

Good (idempotent):
  DELETE FROM predictions WHERE run_date = '2024-01-15'
  INSERT INTO predictions SELECT ... WHERE run_date = '2024-01-15'
  # If retried: old rows deleted, fresh rows inserted — same result

Or use UPSERT (INSERT OR REPLACE): always idempotent.

Rule of thumb: design every Airflow task so it can be safely retried 10 times."""},

    {"q": "How does Airflow's scheduler work?",
     "a": """The Airflow scheduler:
1. Parses all DAG files in the dags/ folder (every 30 seconds by default)
2. Checks which DAG runs are due based on schedule_interval and last successful run
3. Creates DagRun objects for due DAGs
4. Creates TaskInstance objects for each task
5. Uses topological sort to find tasks with no unfinished dependencies
6. Submits ready tasks to the executor (LocalExecutor, CeleryExecutor, KubernetesExecutor)
7. Executor runs tasks and reports status back
8. Scheduler updates task states and finds next ready tasks

Executors:
- SequentialExecutor: one task at a time (default, for testing)
- LocalExecutor: parallel tasks on one machine
- CeleryExecutor: distributed tasks across many workers
- KubernetesExecutor: each task in its own K8s pod"""},
]

for i, item in enumerate(qa, 1):
    print(f"Q{i}: {item['q']}")
    print(f"A:  {item['a'].strip()}")
    print("-" * 65)
    print()

## Summary

| Concept | Key Detail |
|---------|------------|
| DAG | Python object defining a workflow; runs on a schedule |
| Operator | A single task type (Python, Bash, SQL, HTTP, etc.) |
| Task Instance | One execution of one operator for one DAG run |
| XCom | Small data passed between tasks (max ~48KB in SQLite) |
| Schedule | Cron expression or preset (@daily, @hourly) |
| Executor | How tasks are run (local, Celery, Kubernetes) |
| Catchup | Whether to run past-due DAG runs on startup |
| Retry | Automatic re-run on failure (configure retries + retry_delay) |

### Next Steps
1. **Install locally**: `pip install apache-airflow && airflow standalone`
2. **Airflow tutorial**: [https://airflow.apache.org/docs/apache-airflow/stable/tutorial/](https://airflow.apache.org/docs/apache-airflow/stable/tutorial/)
3. **Astronomer guides** (best Airflow tutorials): [https://www.astronomer.io/guides/](https://www.astronomer.io/guides/)
4. **Next**: Learn DVC for versioning the datasets and models your Airflow pipelines produce